In [16]:
!pip install PySastrawi scikit-learn pandas numpy joblib

In [17]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("Semua library berhasil diimport!")

Semua library berhasil diimport!


In [18]:
# Baca dengan separator titik koma dan encoding UTF-8-sig (hapus BOM otomatis)
df1 = pd.read_csv('../data/dataset_kompas_4k.csv', sep=';', encoding='utf-8-sig', on_bad_lines='skip')
df2 = pd.read_csv('../data/dataset_tempo_6k.csv', sep=';', encoding='utf-8-sig', on_bad_lines='skip')
df3 = pd.read_csv('../data/dataset_turnbackhoax_10k.csv', sep=';', encoding='utf-8-sig', on_bad_lines='skip')
df4 = pd.read_csv('../data/dataset_cnn_10k.csv', sep=';', encoding='utf-8-sig', on_bad_lines='skip')

print("Kolom Kompas:", df1.columns.tolist())
print("Shape Kompas:", df1.shape)
print("\nKolom TurnBackHoax:", df3.columns.tolist())
print("Shape TurnBackHoax:", df3.shape)

# Cek nilai unik Tags di TurnBackHoax (ini yang berlabel hoax)
print("\nContoh Tags TurnBackHoax:")
print(df3['Tags'].value_counts().head(10))

print("\nContoh Tags Kompas:")
print(df1['Tags'].value_counts().head(10))

Kolom Kompas: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
Shape Kompas: (4750, 6)

Kolom TurnBackHoax: ['Title', 'Timestamp', 'FullText', 'Tags', 'Author', 'Url']
Shape TurnBackHoax: (10390, 6)

Contoh Tags TurnBackHoax:
Tags
Fitnah;Hasut;Hoax                                                                                                        9723
Lain-lain                                                                                                                 577
Berita                                                                                                                     76
Events                                                                                                                      2
https://turnbackhoax.id/2018/08/15/salah-pengakuan-jusuf-kalla-perjuangan-moral-jokowi/                                     1
https://turnbackhoax.id/2019/01/23/salah-artikel-sangat-berharga-pengakuan-jujur-jusuf-kalla-perjuangan-moral-jokowi/       1
ht

In [19]:
# Beri label
df1['label'] = 0  # Kompas = valid
df2['label'] = 0  # Tempo = valid
df3['label'] = 1  # TurnBackHoax = hoax
df4['label'] = 0  # CNN = valid

# Gabungkan semua dataset
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Pakai kolom Title + FullText sebagai fitur teks
df['text'] = df['Title'].fillna('') + ' ' + df['FullText'].fillna('')

# Hapus baris yang teks-nya kosong
df = df[df['text'].str.strip() != '']
df = df[['text', 'label']].dropna()

print("Total data:", df.shape)
print("\nDistribusi label:")
print(df['label'].value_counts())
print("\nContoh data:")
print(df[['text', 'label']].head(3))

# Ambil sampel valid sebanyak data hoax supaya seimbang
df_valid = df[df['label'] == 0].sample(n=10390, random_state=42)
df_hoax = df[df['label'] == 1]

df = pd.concat([df_valid, df_hoax], ignore_index=True).sample(frac=1, random_state=42)

print("Distribusi setelah diseimbangkan:")
print(df['label'].value_counts())
print("Total data:", df.shape)

Total data: (21711, 2)

Distribusi label:
label
0    11321
1    10390
Name: count, dtype: int64

Contoh data:
                                                text  label
0  Efek Ekor Jas Pencalonan Anies, Elektabilitas ...      0
1  Ekonomi 2024 Ditargetkan Tumbuh 5,7 Persen, pa...      0
2  Survei Litbang Kompas: PDI-P, Gerindra, dan Go...      0
Distribusi setelah diseimbangkan:
label
0    10390
1    10390
Name: count, dtype: int64
Total data: (20780, 2)


In [20]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi Sastrawi
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()
factory_stopword = StopWordRemoverFactory()
stopword_remover = factory_stopword.create_stop_word_remover()

def preprocess_text(text):
    # 1. Case Folding
    text = str(text).lower()
    # 2. Cleaning
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    # 4. Stopword Removal
    text = stopword_remover.remove(text)
    # 5. Stemming
    text = stemmer.stem(text)
    return text

print("Memulai preprocessing... harap tunggu 10-15 menit")
df['text_clean'] = df['text'].apply(preprocess_text)
print("Preprocessing selesai!")
print(df[['text', 'text_clean']].head(3))

Memulai preprocessing... harap tunggu 10-15 menit
Preprocessing selesai!
                                                    text  \
803    The Yudhoyono Institute dan Prospek AHY di Pan...   
20402  [KLARIFIKASI] GAMBAR ANAK ALLEPO TERLANTAR SUM...   
19425  [BERITA] “Teror Kiai di Ponpes Kediri Hoax, In...   

                                              text_clean  
803    the yudhoyono institute prospek ahy panggung p...  
20402  klarifikasi gambar anak allepo lantar sumber m...  
19425  berita teror kiai ponpes diri hoax cerita akib...  


In [21]:
from sklearn.model_selection import train_test_split

X = df['text_clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data latih: {X_train.shape[0]} baris")
print(f"Data uji  : {X_test.shape[0]} baris")

Data latih: 16624 baris
Data uji  : 4156 baris


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF selesai, shape:", X_train_tfidf.shape)

TF-IDF selesai, shape: (16624, 5000)


In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("=" * 40)
print("NAIVE BAYES")
print("=" * 40)
print(f"Accuracy : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_nb, average='weighted'):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_nb, average='weighted'):.4f}")
print(classification_report(y_test, y_pred_nb))

NAIVE BAYES
Accuracy : 0.9728
Precision: 0.9729
Recall   : 0.9728
F1-Score : 0.9728
              precision    recall  f1-score   support

           0       0.98      0.97      0.97      2078
           1       0.97      0.98      0.97      2078

    accuracy                           0.97      4156
   macro avg       0.97      0.97      0.97      4156
weighted avg       0.97      0.97      0.97      4156



In [24]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(random_state=42, max_iter=1000)
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

print("=" * 40)
print("SUPPORT VECTOR MACHINE (SVM)")
print("=" * 40)
print(f"Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_svm, average='weighted'):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_svm, average='weighted'):.4f}")
print(classification_report(y_test, y_pred_svm))

SUPPORT VECTOR MACHINE (SVM)
Accuracy : 0.9969
Precision: 0.9969
Recall   : 0.9969
F1-Score : 0.9969
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2078
           1       1.00      1.00      1.00      2078

    accuracy                           1.00      4156
   macro avg       1.00      1.00      1.00      4156
weighted avg       1.00      1.00      1.00      4156



In [25]:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(svm_model, 'models/hoax_model.pkl')
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
print("Model berhasil disimpan!")

Model berhasil disimpan!
